# Sunside + Sola Face LoRA

Генерація з навченою LoRA після `sola_face_lora_colab`.

1. **Runtime → GPU** → Restart session
2. Запусти клітинки **по черзі** (не Run all — спочатку upload LoRA)
3. У UI:
   - Character → **Sola**
   - LoRA → `sola_face_sdxl` weight **0.8–1.0**
   - Prompt: `sola_face,` + сцена (без опису обличчя)

Файл з тренування: `sola_face_sdxl.safetensors` (+ опційно `000004` / `000002`)


In [ ]:
# @title 1) Клон Sunside
import os
import shutil
import subprocess
import sys
import time
import zipfile

os.chdir("/content")
REPO = "/content/Fooocus"
REPO_URL = "https://github.com/sunsideaspect/foocus_sunside.git"
ZIP_URL = "https://github.com/sunsideaspect/foocus_sunside/archive/refs/heads/main.zip"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pygit2==1.15.1"], check=True)


def remove_repo(path: str) -> None:
    if os.path.exists(path):
        subprocess.run(["rm", "-rf", path], check=False)


def repo_ok(path: str) -> bool:
    return os.path.isfile(os.path.join(path, "launch.py"))


def clone_with_git() -> bool:
    remove_repo(REPO)
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO],
        capture_output=True,
        text=True,
        cwd="/content",
    )
    if r.returncode != 0:
        print("git stderr:", (r.stderr or "").strip())
        return False
    return repo_ok(REPO)


def clone_with_zip() -> bool:
    remove_repo(REPO)
    zip_path = "/content/foocus_sunside_main.zip"
    r = subprocess.run(["wget", "--progress=dot:giga", "-O", zip_path, ZIP_URL], cwd="/content")
    if r.returncode != 0:
        return False
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("/content")
    extracted = "/content/foocus_sunside-main"
    if not os.path.isdir(extracted):
        return False
    shutil.move(extracted, REPO)
    return repo_ok(REPO)


ok = False
for attempt in range(1, 4):
    if clone_with_git():
        ok = True
        print(f"✅ Sunside (git, {attempt})")
        break
    print(f"retry git {attempt}/3...")
    time.sleep(8)

if not ok:
    ok = clone_with_zip()
    if ok:
        print("✅ Sunside (ZIP)")

if not ok:
    raise RuntimeError("Не вдалось клонувати. Restart session → знову.")

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu.returncode != 0 or "GPU" not in (gpu.stdout or ""):
    raise RuntimeError("❌ GPU немає. Runtime → GPU")
print("✅", (gpu.stdout or "").strip().split("\n")[0])
print("→ Далі: клітинка 2 — upload LoRA")


In [ ]:
# @title 2) Upload sola_face LoRA
import os
import shutil
from google.colab import files

REPO = "/content/Fooocus"
LORA_DIR = os.path.join(REPO, "models", "loras")
os.makedirs(LORA_DIR, exist_ok=True)

# --- варіант A: файл уже в Drive ---
DRIVE_LORA = ""  # наприклад: /content/drive/MyDrive/sola_face_sdxl.safetensors

if DRIVE_LORA.strip():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    src = DRIVE_LORA.strip()
    if not os.path.isfile(src):
        raise FileNotFoundError(src)
    dst = os.path.join(LORA_DIR, os.path.basename(src))
    shutil.copy2(src, dst)
    print("✅ з Drive →", dst, os.path.getsize(dst))
else:
    print("Обери .safetensors (sola_face_sdxl.safetensors і/або 000004 / 000002)")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Нічого не завантажено")
    for name in uploaded:
        if not name.lower().endswith(".safetensors"):
            print("⚠️ skip (не safetensors):", name)
            continue
        src = os.path.join("/content", name)
        # files.upload кладе в cwd; на Colab часто /content
        if not os.path.isfile(src):
            src = name
        dst = os.path.join(LORA_DIR, os.path.basename(name))
        shutil.move(src, dst)
        print("✅", dst, f"({os.path.getsize(dst)/1e6:.1f} MB)")

print("\nLoRAs у", LORA_DIR)
for fn in sorted(os.listdir(LORA_DIR)):
    if fn.endswith(".safetensors"):
        print(" -", fn)

need = [f for f in os.listdir(LORA_DIR) if "sola_face" in f.lower() and f.endswith(".safetensors")]
if not need:
    print("⚠️ Немає файла з sola_face у назві — все одно можна вибрати в UI")
else:
    print("\nГотово. Trigger у prompt: sola_face,")
    print("→ Далі: клітинка 3 — запуск")


In [ ]:
# @title 3) Preset
SELECTED_PRESET = "realistic_cyberrealistic_xl"
model_choice = "CyberRealistic XL"
print(f"✅ Preset: {SELECTED_PRESET}")
print("Character → Sola | LoRA sola_face_sdxl @ 0.8–1.0 | prompt: sola_face, ...")


In [ ]:
# @title 4) Launch Sunside
import os
import re
import stat
import subprocess
import sys

from IPython.display import HTML, display

if "SELECTED_PRESET" not in globals():
    SELECTED_PRESET = "realistic_cyberrealistic_xl"
    model_choice = "CyberRealistic XL"

REPO = "/content/Fooocus"
if not os.path.isfile(os.path.join(REPO, "launch.py")):
    raise RuntimeError("Спочатку клітинка 1 (клон).")
os.chdir(REPO)

os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
os.environ["LAUNCH_LIVE_OUTPUT"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["SUNSIDE_PRODUCT"] = "1"
os.environ["SUNSIDE_FACELOCK_CPU"] = "1"

try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
        print("✅ HF_TOKEN")
except Exception:
    pass


def gpu_total_vram_mb() -> int:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
        check=True,
    )
    return int((r.stdout or "").strip().split("\n")[0])


vram_mb = gpu_total_vram_mb()
vram_flag = "--always-high-vram" if vram_mb >= 22000 else "--always-normal-vram"
print(f"VRAM {vram_mb} MB → {vram_flag}")


def _bar_html(pct: int, detail: str) -> str:
    pct = max(0, min(100, pct))
    safe = detail.replace("&", "&amp;").replace("<", "&lt;")
    return (
        f'<div style="margin:6px 0 10px">'
        f'<div style="background:#1e293b;border-radius:8px;height:26px;overflow:hidden;border:1px solid #334155">'
        f'<div style="width:{pct}%;height:100%;background:linear-gradient(90deg,#16a34a,#4ade80);'
        f'transition:width .4s"></div></div>'
        f'<div style="font:12px/1.4 monospace;color:#94a3b8;margin-top:4px">{pct}% — {safe}</div></div>'
    )


def _emit_progress(line: str, bar_holder: dict) -> bool:
    m = re.search(r"(\d+)%", line)
    if not m or not ("|" in line or "G/" in line or "M/" in line or "kB" in line):
        return False
    pct = int(m.group(1))
    html = _bar_html(pct, line.strip()[:140])
    if bar_holder.get("disp") is None:
        bar_holder["disp"] = display(HTML(html), display_id=True)
    else:
        bar_holder["disp"].update(HTML(html))
    return True


def run_live(argv, cwd=REPO, label=""):
    if label:
        print(label, flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["LAUNCH_LIVE_OUTPUT"] = "1"
    p = subprocess.Popen(
        argv,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    assert p.stdout is not None
    bar_holder: dict = {}
    partial = b""
    last_lines: list = []
    while True:
        chunk = p.stdout.read(1)
        if not chunk:
            if p.poll() is not None:
                break
            continue
        if chunk in (b"\r", b"\n"):
            line = partial.decode("utf-8", errors="ignore").strip()
            partial = b""
            if not line:
                continue
            last_lines.append(line)
            if len(last_lines) > 40:
                last_lines.pop(0)
            if not _emit_progress(line, bar_holder):
                print(line, flush=True)
        else:
            partial += chunk
    tail = partial.decode("utf-8", errors="ignore").strip()
    if tail:
        last_lines.append(tail)
        if not _emit_progress(tail, bar_holder):
            print(tail, flush=True)
    rc = p.wait()
    if rc != 0:
        tip = "\n--- log ---\n" + "\n".join(last_lines[-25:])
        if rc in (-9, 137):
            raise RuntimeError("OOM (-9). Restart session → знову." + tip)
        raise RuntimeError(f"exit {rc}: {' '.join(argv)}" + tip)


print("→ numpy / gradio...")
run_live([sys.executable, "-m", "pip", "install", "--force-reinstall", "numpy==1.26.4"], cwd="/content")
run_live(
    [sys.executable, "-m", "pip", "install",
     "starlette>=0.27.0,<1.0.0", "fastapi>=0.100.0,<0.115.0", "gradio==3.41.2"],
    cwd=REPO,
)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "hf_xet", "hf-xet"], check=False)


def gradio_pkg_dir() -> str:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "show", "gradio"],
        capture_output=True,
        text=True,
        check=True,
    )
    for line in r.stdout.splitlines():
        if line.startswith("Location:"):
            return os.path.join(line.split(":", 1)[1].strip(), "gradio")
    raise RuntimeError("gradio not found")


frpc = os.path.join(gradio_pkg_dir(), "frpc_linux_amd64_v0.2")
if not (os.path.isfile(frpc) and os.path.getsize(frpc) > 1_000_000):
    subprocess.run(
        ["wget", "--progress=dot:giga", "-O", frpc,
         "https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_amd64"],
        check=True,
    )
    os.chmod(frpc, stat.S_IRWXU)

lora_dir = os.path.join(REPO, "models", "loras")
solas = [f for f in os.listdir(lora_dir) if "sola_face" in f.lower() and f.endswith(".safetensors")] if os.path.isdir(lora_dir) else []
print("Sola LoRAs:", solas or "(немає — upload у клітинці 2)")

launch_args = [
    sys.executable, "-u", "launch.py",
    "--preset", SELECTED_PRESET,
    "--disable-censor",
    "--disable-pro-mode",
    "--disable-preset-selection",
    "--share",
    vram_flag,
    "--disable-in-browser",
]

print("\nЗАПУСК… чекай public URL\n")
os.chdir(REPO)
run_live(launch_args, cwd=REPO)
